# Step 2: Generate summary data...with functions 😎

Generate summary data as html tables
* create summary for Lichert scale responses (i.e., pivot table of responses)
* get comments
* split topics (comma-separated responses)
* combine summary data

## 1. Set variables

In [ ]:
# files to open
access_eval = 'ACCESS Advising Evaluation (Responses).xlsx'
soc_eval = 'SOC Advising (Responses).xlsx'

# time variables
filter_year = 2025
filter_month = 11

# questions to remove
# November 2025 survey questions changed and Google Sheets was reindexed.
# Starting December 2025, questions do not need to be removed.

# text to remove from each question
eval_remove_text = ['Please evaluate your advising session',
                    'Please evaluate the advisor',
                    'Overall advising appointment',
                    '[',
                    ']',
                    '.',
                    ':',
                    'Check all that apply.']

# text to replace
comments_replace = ['Please feel free to make additional comments of the above questions', 'Additional comments?']

# Likert scale values
likert_scale_access1 = ['Strongly Agree (5)', 'Agree (4)', 'Neutral (3)', 'Disagree (2)', 'Strongly Disagree (1)', 'Not Applicable (0)', 'No Response']
likert_scale_access2 = ['Excellent (5)', 'Good (4)', 'Average (3)', 'Fair (2)', 'Poor (1)', 'No Response']
likert_scale_soc1 = ['Strongly agree', 'Agree', 'Neutral', 'Disagree', 'Strongly Disagree', 'No Response']
likert_scale_soc2 = ['Excellent', 'Good', 'Average', 'Fair', 'Poor', 'No Response']

# meeting types
meeting_types = ['Via email', 'Via phone', 'Via Zoom', 'In person']

## 2. Load and clean data (from Step 1)

In [ ]:
# load libraries
import pandas as pd
from IPython.display import display, HTML  # library to display html in notebook

In [ ]:
# load data
access_df = pd.read_excel(access_eval)
soc_df = pd.read_excel(soc_eval)

In [ ]:
# clean data
def clean_data(df, text_remove_list, text_replace_list):
    for i in text_remove_list:
        df.columns = df.columns.str.replace(i, '')

    for i in text_replace_list:
        df.columns = df.columns.str.replace(i, 'Comments')

    df.columns = df.columns.str.strip()

    df = df[df['Timestamp'].dt.year == filter_year]
    df = df[df['Timestamp'].dt.month == filter_month]

    df = df.fillna('No Response')
    
    return df

# clean data
access_df = clean_data(access_df, eval_remove_text, comments_replace)
soc_df = clean_data(soc_df, eval_remove_text, comments_replace)

In [ ]:
# check data
soc_df

## 3. Filter data for a single advisor

In [ ]:
# filter data for a single advisor
#advisor = 'Marilou'
#filtered_advisor = access_df[access_df['Advisor'].str.startswith(advisor)]

def filtered_advisor_df (advisor_name, df_name):
    filtered_advisor = df_name[df_name['Advisor'].str.startswith(advisor_name)]

    return filtered_advisor

In [ ]:
# check data
#filtered_advisor
#filtertest

## 4. Functions to generate summary data

In [ ]:
# function to create summary data for Likert scale responses
# def function_name(var1, var2):
#    code to create summary data
#    return html_table

def create_summary_data(col_beg, col_end, scale, filtered_advisor):
    question_score = []

    # get list of questions
    questions = []
    questions = filtered_advisor.columns[col_beg:col_end]

    number_responses = len(filtered_advisor)

    for i in questions:
        for j in scale:
            count = (filtered_advisor[i] == j).sum()
            percent = count / number_responses * 100
            display_count = f'{count}<br>{percent:.1f}%'
            question_score.append({
                'Question': i,
                'Score': j,
                'Count': display_count
            })

    df = pd.DataFrame(question_score)
    df = df.pivot(index='Question', columns='Score', values='Count')

    df = df[scale].loc[questions]
    df.columns.name = None

    html_likert = df.to_html(escape=False)
    
    return html_likert

In [ ]:
# function to get comments
# def function_name(var1, var2):
#    code to get comments
#    return html_table

def create_comments_data(col_beg, col_end, filtered_advisor):
    comments = filtered_advisor.iloc[:, col_end]

    comments = comments[comments != 'No Response']

    commentsdf = pd.DataFrame(comments)

    html_comments = commentsdf.to_html(index=False)

    if len(html_comments) < 1:
        html_comments = ''

    return html_comments

In [ ]:
# function to split comma-separated responses
# def function_name(var1, var2):
#    code to split responses
#    return html_table

def soc_split_topics(meeting_col, rename_col, filtered_advisor):
    soc_topics = []

    meeting = filtered_advisor.columns[meeting_col]
    responses = filtered_advisor[meeting].values

    for i in responses:
        # replace response that contains a comma
        i = i.replace('Major information, options', 'Major information/options')
        i = i.replace(', ', ',')

        # split values by comma
        split = i.split(',')

        # add topic to list
        for j in split:
            soc_topics.append(j)

    df = pd.DataFrame(soc_topics)

    summary = df.value_counts().reset_index()
    summary = summary.rename(columns={rename_col: 'Topics'})

    html_topics = summary.to_html(index=False)

    html_topics = html_topics + '<p>One student can select more than one topic during a meeting.</p>'

    return html_topics

In [ ]:
# meeting types function for advisor

def soc_meeting_types(meeting_type_col, filtered_advisor):
    soc_meeting_types = []

    # 2 column
    meeting_types = filtered_advisor.column[meeting_type_col]
    meeting_types_values = filtered_advisor[meeting_types].values

    return meeting_types_values

soctype = 

In [ ]:
# function to combine summary data (put html tables together)
# def function_name(var1, var2):
#    code to combine html tables together
#    return html_final

def combine_summary_data(df_name, html1, html2, html3):
    if df_name == 'soc_df':
        html_summary_data = html3 + html1 + html2
    else:
        html_summary_data = html1 + html2

    return html_summary_data

In [ ]:
# call function

# loop through both lists, shift the columns for each type of likert scales
# loop for each advisor

html_file = ''
html_topics = ''

# filtered_advisor = filtered_advisor_df ('Marilou', access_df)

# accessdf
advisors_list = access_df['Advisor'].unique()

for i in advisors_list:
    filtered_advisor = filtered_advisor_df(i, access_df)

    # likerts
    html_likert = create_summary_data(2, 4, likert_scale_access1, filtered_advisor)
    html_likert = html_likert + create_summary_data(9, 10, likert_scale_access2, filtered_advisor)

    # comments
    html_comments = create_comments_data(0, -1, filtered_advisor)

    # combined summary_data
    html_summary_data = combine_summary_data('access_df', html_likert, html_comments, html_topics)

    html_file = html_file + html_likert + html_comments + html_summary_data

# socdf
advisors_list = soc_df['Advisor'].unique()

for i in advisors_list:
    filtered_advisor = filtered_advisor_df(i, soc_df)

    # likerts
    html_likert = create_summary_data(4, 5, likert_scale_soc1, filtered_advisor)
    html_likert = html_likert + create_summary_data(6, 7, likert_scale_soc2, filtered_advisor)

    # comments
    html_comments = create_comments_data(0, -1, filtered_advisor)

    # topics
    html_topics = soc_split_topics(3, 0, filtered_advisor)

    # combined summary_data
    html_summary_data = combine_summary_data('access_df', html_likert, html_comments, html_topics)

    html_file = html_file + html_likert + html_comments + html_summary_data


#display(HTML(html_file))



# check data


## 👏 You made really nice functions, yay!